# Sentinel-1 and Sentinel-2 parcel zonal statistics

Configure and run `SatelliteZonalStats`. openEO creates one cloud-masked, indexed monthly NetCDF cube per spatial batch. IQR cleaning, null filling, and parcel statistics run locally, producing one ML-ready GeoParquet row per parcel with dated feature columns.

In [1]:
from pathlib import Path

from openeo_parcel_stats_pipeline.zonal_stats_multiuser_manager import (
    split_geodataframe_by_grid,
    load_openeo_users_from_env,
    run_parallel_extractions,
    merge_and_save_results,
)
from openeo_parcel_stats_pipeline.zonal_stats_job_manager import (
    compute_tile_width,
    compute_tile_buffer_metres,
)
from common_libraries.io_library import read_data
import common_libraries.geom_library as geom_l

c:\ProgramData\anaconda3\envs\axhub_spatial\Lib\site-packages\xicor\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


## Inputs

Processing order:

`Sentinel-2 L2A`
`acquisition cloud mask`
`acquisition pixel indices`
`monthly pixel median`
`monthly per-variable cube IQR outlier removal`
`local null filling`
`parcel statistics`
`one-row-per-parcel time-series pivot`
`ML-ready GeoParquet`

Temporal filling is past-only so later observations cannot leak into earlier feature periods.

In [2]:
WORK_PATH = Path("C:/work_dir")
RUN_IDENTIFIER = "volvi-2023-2024"  # Change this for each named run.

PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "parcels_volvi.gpkg"

In [3]:
gdf_parcels = read_data(str(PARCELS_PATH))
gdf_parcels = geom_l.return_valid_geometries(gdf_parcels)
gdf_parcels = geom_l.convert_multipolygons_to_polygons(gdf_parcels)
print(len(gdf_parcels), "parcels in study area")
gdf_parcels.head(3)

Reading file: C:/work_dir/volvi-2023-2024/parcels_volvi.gpkg
28699 parcels in study area


,parcel_code,initial_label_code,initial_label,predicted_label_code,predicted_label,prediction_probability,set_type,geometry
0,51480984,110,wheat,119,grass_fodders,0.3478126800,train,POLYGON ((423723...
1,237535570,110,wheat,110,wheat,0.6774292600,train,POLYGON ((412455...
2,237535579,110,wheat,110,wheat,0.2981949200,test,POLYGON ((412401...


In [4]:
label_translations = {
    "olive_groves": "ελαιώνες",
    "grape_vines": "αμπελώνες",
    "pome_trees": "ροδιές",
    "stone_fruit_trees": "πυρηνόκαρπα",
    "other_trees": "λοιπές δενδρώδεις καλλιέργειες",
    "kiwi": "ακτινίδια",
    "nut_trees": "καρυδιές",
    "rice": "ρύζι",
    "wheat": "σιτάρι",
    "barley": "κριθάρι",
    "other_cereals": "λοιπά σιτηρά",
    "cotton": "βαμβάκι",
    "maize": "αραβόσιτος",
    "tobacco": "καπνός",
    "sunflower": "ηλίανθος",
    "rapeseed": "ελαιοκράμβη",
    "grass_fodders": "χορτοδοτικές καλλιέργειες",
    "beans": "φασόλια",
    "legumes": "όσπρια",
    "potato": "πατάτα",
    "tomato": "ντομάτα",
    "melons": "πεπονοειδή",
    "horticultural_fall": "φθινοπωρινά κηπευτικά",
    "horticultural_spring": "ανοιξιάτικα κηπευτικά",
    "horticultural_summer": "θερινά κηπευτικά",
    "aromatic": "αρωματικά φυτά",
    "fallow": "αγρανάπαυση",
    "pastures": "βοσκότοποι",
    "greenhouses": "θερμοκήπια",
}

df_classes = gdf_parcels[["initial_label_code", "initial_label"]].copy()
df_classes = df_classes.drop_duplicates().sort_values("initial_label_code")
df_classes["initial_label_gr"] = (
    df_classes["initial_label"].map(label_translations).fillna(df_classes["initial_label"])
)
df_classes.reset_index(drop=True)

,initial_label_code,initial_label,initial_label_gr
0,101,olive_groves,ελαιώνες
1,102,grape_vines,αμπελώνες
2,104,pome_trees,ροδιές
3,105,stone_fruit_trees,πυρηνόκαρπα
4,106,other_trees,λοιπές δενδρώδει...
5,107,kiwi,ακτινίδια
6,108,nut_trees,καρυδιές
7,110,wheat,σιτάρι
8,111,barley,κριθάρι
9,112,other_cereals,λοιπά σιτηρά


In [5]:
gdf_parcels["label_gr"] = gdf_parcels["initial_label"].map(label_translations)
gdf_parcels.head(3)

,parcel_code,initial_label_code,initial_label,predicted_label_code,predicted_label,prediction_probability,set_type,geometry,label_gr
0,51480984,110,wheat,119,grass_fodders,0.3478126800,train,POLYGON ((423723...,σιτάρι
1,237535570,110,wheat,110,wheat,0.6774292600,train,POLYGON ((412455...,σιτάρι
2,237535579,110,wheat,110,wheat,0.2981949200,test,POLYGON ((412401...,σιτάρι


In [6]:
class_column = "label_gr"
parcel_id_column = "parcel_code"

In [7]:
all_gdf = gdf_parcels.copy()
all_gdf["accuracy"] = all_gdf["initial_label_code"] == all_gdf["predicted_label_code"]
accuracy = all_gdf["accuracy"].mean()
print(f"Accuracy on all data: {accuracy:.2%}")

train_gdf = gdf_parcels[gdf_parcels["set_type"]=="train"].copy()
print(len(train_gdf), "rows in train_gdf after filtering with set_type=='train'")
train_gdf["accuracy"] = train_gdf["initial_label_code"] == train_gdf["predicted_label_code"]
accuracy = train_gdf["accuracy"].mean()
print(f"Accuracy on train data: {accuracy:.2%}")

test_gdf = gdf_parcels[gdf_parcels["set_type"]=="test"].copy()
print(len(test_gdf), "rows in test_gdf after filtering with set_type=='test'")
test_gdf["accuracy"] = test_gdf["initial_label_code"] == test_gdf["predicted_label_code"]
accuracy = test_gdf["accuracy"].mean()
print(f"Accuracy on test data: {accuracy:.2%}")

Accuracy on all data: 57.88%
11489 rows in train_gdf after filtering with set_type=='train'
Accuracy on train data: 71.89%
17210 rows in test_gdf after filtering with set_type=='test'
Accuracy on test data: 48.53%


In [8]:
START_DATE = "2023-10-01"
END_DATE = "2024-09-30"
OUTPUT_DIR = WORK_PATH / RUN_IDENTIFIER / "1_parcel_stats"
WORKING_EPSG = 32635

CALCULATE_SENTINEL2_INDICES = True
SENTINEL2_INDICES = ["EVI", "NDMI", "NDRE", "NDVI", "NDWI",  "SAVI"]
SENTINEL2_BANDS = ["B02", "B03", "B04", "B05", "B08", "B11", "B12"]
SENTINEL1_BANDS = ["VV", "VH"]

# Mean is mandatory; percentiles enable derived IQR and robust skewness.
SPATIAL_STATISTICS = [
    "mean",
    "median",
    "sd",
    "min",
    "max",
    # "count",
    "p10",
    "p25",
    "p75",
    "p90",
]

INTERPOLATION_METHOD = "nearest"  # "idw"
INTERPOLATION_MAX_DISTANCE_IN_METERS = 500

# IQR outliers become null, then nulls are filled.
REMOVE_OUTLIERS = True
IQR_QUANTILES = (0.1, 0.9)
IQR_MULTIPLIER = 1.5
FILL_NULLS = True
IQR_MIN_VALID_PIXELS = 20
TEMPORAL_FILL_MODE = "bidirectional" # or past_only
MINIMUM_PARCEL_PIXELS = 3
MINIMUM_OBSERVED_FRACTION_FOR_FILL = 0.20

# Each worker loads one complete monthly cube; use one unless RAM has been sized for parallel cubes.
LOCAL_BATCH_WORKERS = 1

# Set CRS to WGS84 if missing, then reproject to the working CRS.
if gdf_parcels.crs is None:
    print("WARNING: No CRS found on parcels GeoDataFrame. Assuming EPSG:4326 (WGS84).")
    gdf_parcels = gdf_parcels.set_crs(epsg=4326)

gdf_parcels = gdf_parcels.to_crs(WORKING_EPSG)


## Tile-size recommendation

This local comparison selects the smallest candidate grid whose buffered ownership tiles fully cover every parcel.

In [9]:
TILE_SIZE_METRES = compute_tile_width(gdf_parcels, parcel_id_col=parcel_id_column, working_epsg=WORKING_EPSG)
TILE_BUFFER_METRES = compute_tile_buffer_metres(gdf_parcels, WORKING_EPSG)

AOI extent: 50.47 x 33.15 km; parcels: 28,699
 tile_size_km  non_empty_jobs  crossing_parcels  median_parcels_per_job  max_parcels_per_job  loaded_area_km2
            5              41              1223          511.0000000000                 2152   965.3475326561
           10              15               587         1252.0000000000                 6514  1267.7182690504
           20               4               239         8851.5000000000                10069  1272.4597418388
           30               3               101         4619.0000000000                23316  1608.6896127583
           40               2                15        14342.0000000000                27996  1673.1594228491
           50               2                 0        14349.5000000000                28695  1673.1594228491
           60               1                 0        28699.0000000000                28699  1673.1594228491
Recommended tile size: 50 km
Recommended tile buffer: 1,800 m


## Run extraction

In [10]:
TILE_SIZE_METRES = 20_000
TILE_BUFFER_METRES = 2_000

In [11]:
# ============================================================================
# STEP 1: Split parcels into spatial partitions
# ============================================================================
# This divides the parcels into a grid (1 row, 2 columns = left/right halves)
# For 5 users, use: rows=1, cols=5

spatial_parts = split_geodataframe_by_grid(gdf=gdf_parcels, rows=2, cols=2)

print(f"Created {len(spatial_parts)} spatial partitions:")
for i, part in enumerate(spatial_parts, 1):
    print(f"  Partition {i}: {len(part)} parcels")

Spatial grid: 2x2 cells
  Bounds: X=[152249, 202723], Y=[4503044, 4536192]
  Cell size: 25237m x 16574m
  Cell (0,0): 8506 parcels
  Cell (0,1): 7266 parcels
  Cell (1,0): 12927 parcels
  Cell (1,1): (empty)
Created 3 spatial partitions:
  Partition 1: 8506 parcels
  Partition 2: 7266 parcels
  Partition 3: 12927 parcels


In [12]:
# ============================================================================
# STEP 2: Load openEO users from environment variable
# ============================================================================
# Requires OPENEO_USERS env var: "user1@example.com,pass1;user2@example.com,pass2"

users_list = load_openeo_users_from_env()

print(f"Loaded {len(users_list)} openEO user(s):")
for i, (user, _) in enumerate(users_list, 1):
    print(f"  User {i}: {user}")


Loaded 5 openEO user(s): ['louvarieleni@gmail.com', 'farfarasmpoytos@gmail.com', 'esagovhub1@gmail.com', 'esagovhub2@gmail.com', 'esagovhub3@gmail.com']
Each spatial partition will use one user (round-robin assignment)
Loaded 5 openEO user(s):
  User 1: louvarieleni@gmail.com
  User 2: farfarasmpoytos@gmail.com
  User 3: esagovhub1@gmail.com
  User 4: esagovhub2@gmail.com
  User 5: esagovhub3@gmail.com


In [ ]:
# ============================================================================
# STEP 3: Create extraction configuration dictionary
# ============================================================================
# Single dictionary containing all parameters for the extraction process,
# to be passed to each parallel worker.

extraction_config = {
    # Output paths
    "output_dir": OUTPUT_DIR,
    "run_identifier": RUN_IDENTIFIER,

    # Parcel configuration
    "parcel_id_field": parcel_id_column,

    # Temporal bounds
    "start_date": START_DATE,
    "end_date": END_DATE,

    # Spatial configuration
    "working_epsg": WORKING_EPSG,

    # Sentinel-2 configuration
    "sentinel2_bands": SENTINEL2_BANDS,
    "calculate_sentinel2_indices": CALCULATE_SENTINEL2_INDICES,
    "sentinel2_indices": SENTINEL2_INDICES,

    # Sentinel-1 configuration
    "sentinel1_bands": SENTINEL1_BANDS,

    # Statistics to calculate
    "spatial_statistics": SPATIAL_STATISTICS,

    # Tiling configuration
    "tile_size_metres": TILE_SIZE_METRES,
    "tile_buffer_metres": TILE_BUFFER_METRES,

    # Data cleaning (IQR outlier removal)
    "remove_outliers": REMOVE_OUTLIERS,
    "iqr_quantiles": IQR_QUANTILES,
    "iqr_multiplier": IQR_MULTIPLIER,
    "iqr_min_valid_pixels": IQR_MIN_VALID_PIXELS,

    # Null filling
    "fill_nulls": FILL_NULLS,
    "temporal_fill_mode": TEMPORAL_FILL_MODE,
    "minimum_parcel_pixels": MINIMUM_PARCEL_PIXELS,
    "minimum_observed_fraction_for_fill": MINIMUM_OBSERVED_FRACTION_FOR_FILL,

    # Interpolation
    "interpolation_method": INTERPOLATION_METHOD,
    "interpolation_max_distance_in_meters": INTERPOLATION_MAX_DISTANCE_IN_METERS,

    # Parallel processing
    "openeo_parallel_jobs": 2,
    "job_poll_seconds": 30,
    "batch_workers": LOCAL_BATCH_WORKERS,

    # Other options
    "spatial_fill_window_sizes": (3, 5),
}

# ============================================================================
# STEP 4: Run all partitions in parallel
# ============================================================================
# Each partition uses a different user (round-robin assignment)
# All partitions extract simultaneously
run_parallel_extractions(
    spatial_parts=spatial_parts,
    users_list=users_list,
    config=extraction_config,
)

# Each partition result is saved below OUTPUT_DIR/partition_<n>.
print("\nExtraction complete!")

In [13]:
# ============================================================================
# STEP 4: Merge results and save to files
# ============================================================================
# Combines all partition results and removes duplicates
# Saves one file:
#   1. satellite_parcel_time_stats.geoparquet (full results with geometry)

merged_gdf, geoparquet_path = merge_and_save_results(
    output_dir=OUTPUT_DIR,
    parcel_id_column=parcel_id_column,
    working_epsg=WORKING_EPSG
)

Loading partition 1: C:\work_dir\volvi-2023-2024\1_parcel_stats\partition_1\satellite_parcel_time_stats.geoparquet
Loading partition 2: C:\work_dir\volvi-2023-2024\1_parcel_stats\partition_2\satellite_parcel_time_stats.geoparquet
Loading partition 3: C:\work_dir\volvi-2023-2024\1_parcel_stats\partition_3\satellite_parcel_time_stats.geoparquet

Merging 3 partition result(s)

Merged results: 28699 parcels (input: 28699)
Output columns: 2714
Output CRS: EPSG:32635

Saving final merged results
Removing existing file: C:/work_dir/volvi-2023-2024/1_parcel_stats/satellite_parcel_time_stats.geoparquet
Writing file: C:/work_dir/volvi-2023-2024/1_parcel_stats/satellite_parcel_time_stats.geoparquet

Saved final output: C:\work_dir\volvi-2023-2024\1_parcel_stats\satellite_parcel_time_stats.geoparquet


In [14]:
# ============================================================================
# STEP 5: Inspect results
# ============================================================================

print(f"\n{'='*70}")
print("FINAL RESULTS")
print(f"{'='*70}")

print("\nMerged GeoDataFrame:")
print(f"  Rows: {len(merged_gdf)}")
print(f"  Columns: {len(merged_gdf.columns)}")
print(f"  Has geometry: {merged_gdf.geometry.name in merged_gdf.columns}")

print("\nOutput files:")
print(f"  GeoParquet: {geoparquet_path}")
print(f"    Size: {geoparquet_path.stat().st_size / 1e6:.1f} MB")

print("\nFirst 5 rows:")
display(merged_gdf.head())


FINAL RESULTS

Merged GeoDataFrame:
  Rows: 28699
  Columns: 2714
  Has geometry: True

Output files:
  GeoParquet: C:\work_dir\volvi-2023-2024\1_parcel_stats\satellite_parcel_time_stats.geoparquet
    Size: 563.6 MB

First 5 rows:


parcel_code  initial_label_code  initial_label  predicted_label_code  \
0   277666994                 119  grass_fodders                  129    
1   277666938                 119  grass_fodders                  111    
2   277666942                 119  grass_fodders                  111    
3   277667008                 110          wheat                  110    
4   277666998                 119  grass_fodders                  110    

  predicted_label  prediction_probability set_type             label_gr  \
0          fallow         0.3141156000       train  χορτοδοτικές καλ...   
1          barley         0.2663752700        test  χορτοδοτικές καλ...   
2          barley         0.1837388100        test  χορτοδοτικές καλ...   
3           wheat         0.4703514000       train               σιτάρι   
4           wheat         0.6944472000        test  χορτοδοτικές καλ...   

              geometry  batch_number  eligible_pixel_count  \
0  POLYGON ((168919...             1                   51    
1  POLYGON ((169127...             1                  284    
2  POLYGON ((169109...             1                   76    
3  POLYGON ((169277...             1                   61    
4  POLYGON ((164148...             1                   71    

   meets_minimum_pixel_count   parcel_area_m2  approx_pixel_count  \
0                 True        4981.3191093498       49.8131910935   
1                 True       28418.2862207056      284.1828622071   
2                 True        7917.8669660429       79.1786696604   
3                 True        5964.7819932065       59.6478199321   
4                 True        7055.0273609518       70.5502736095   

   B02_bowley_skewness__20231001  B02_bowley_skewness__20231101  \
0         0.5117773416                   0.5999997639             
1         0.1031400682                   0.0322586072             
2         0.0798121475                   0.1960771996             
3         0.5203630222                   0.4799996821             
4         0.1780823782                   0.0496456749             

   B02_bowley_skewness__20231201  B02_bowley_skewness__20240101  \
0         0.3978494279                   0.2454546378             
1        -0.0666653421                   0.1333331678             
2        -0.0676714306                   0.0931665636             
3         0.4262294682                   0.1914895303             
4        -0.2324339476                  -0.0458709451             

   B02_bowley_skewness__20240201  B02_bowley_skewness__20240301  \
0         0.5669516885                   0.6146969834             
1        -0.0625010186                   0.1210192775             
2         0.2307692308                   0.0825681677             
3        -0.1627917856                  -0.1199998212             
4         0.2666661148                   0.4841267213             

   B02_bowley_skewness__20240401  B02_bowley_skewness__20240501  \
0         0.7050764318                  -0.2736318113             
1         0.1111108812                   0.1538430678             
2         0.2478635309                   0.1862058476             
3        -0.0645160903                   0.5199992251             
4         0.4615386190                  -0.0133325651             

   B02_bowley_skewness__20240601  B02_bowley_skewness__20240701  \
0        -0.3769234318                  -0.1985817477             
1        -0.0238465934                   0.0482766559             
2         0.0272938907                   0.5614613434             
3         0.0607737081                  -0.0169469266             
4        -0.2594943036                  -0.1538476528             

   B02_bowley_skewness__20240801  B02_bowley_skewness__20240901  \
0        -0.6771299969                   0.3383459489             
1         0.1914895303                   0.0018340737             
2         0.5565610280                  -0.1636364030             
3         0.7333331441             